# Agents-as-Tools

A handoff gives the _whole conversation_ to a specialist. Sometimes you want the
opposite: call the specialist like a tool, get a result back, and keep going as the
original agent. That is `Agent.as_tool()`. Here, the Shadow Clone Sage calls a
Shadow Clone Summoning Specialist as a tool.


In [ ]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)

# Initialize the specialist agent
shadow_clone_agent = Agent(
    name="Shadow Clone Summoning Specialist",
    instructions="Summon a shadow clone from a defeated enemy ninja and report its rank.",
)

# as_tool: wraps the specialist so another agent can call it like a function
summon_tool = shadow_clone_agent.as_tool(
    # tool_name: the name the model sees for this tool
    tool_name="summon_shadow_clone",
    # tool_description: what the tool does, so the model knows when to use it
    tool_description="Summon a shadow clone from a defeated enemy ninja.",
)
# With no parameters=, the tool takes one text field called "input"
print("[default schema]", summon_tool.params_json_schema)

# Initialize the agent that calls the specialist
shadow_clone_sage = Agent(
    name="Shadow Clone Sage",
    instructions="Use summon_shadow_clone whenever an enemy ninja has just been defeated.",
    # tools: the tools this agent is allowed to call
    tools=[summon_tool],
)


# Demo: the Sage calls the specialist as a tool, then writes its own final message
async def demo_as_tool() -> None:
    # Script the model: tool call, then the specialist's reply, then the Sage's reply
    model = ScriptedModel(
        [
            [function_call("summon_shadow_clone", {"input": "A Rogue Ninja was defeated."}, call_id="c1")],
            [assistant_message("Summoning successful: Rogue Ninja (Jonin).")],
            [assistant_message("The Rogue Ninja shadow clone has joined your ranks.")],
        ]
    )

    # Run the Agent
    result = await Runner.run(
        shadow_clone_sage,
        "The Rogue Ninja has been defeated.",
        # run_config: settings for just this run
        run_config=RunConfig(model=model, workflow_name="e09-as-tool-demo"),
    )
    # proves the Sage stays in control: the final message is the Sage's, not the specialist's
    print("[final_output]", result.final_output)


await demo_as_tool()
# Outside a notebook: asyncio.run(demo_as_tool())

## Pass typed input with `parameters=` and `input_builder=`

`parameters=` gives the tool a real typed shape. `input_builder=` turns those typed
arguments into the text the specialist reads. Gotcha: `input_builder` receives a
_plain dict_ at runtime, so use `options["params"]["enemy_name"]`, not
`options.params.enemy_name`.


In [ ]:
from dataclasses import (
    dataclass,  # a decorator that makes a simple class that holds data
)

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Tool parameter shape: the typed arguments the model must fill in for structured_tool below
@dataclass
class ShadowCloneInput:
    """The defeated enemy ninja to summon a shadow clone from."""

    enemy_name: str  # the defeated enemy's name
    enemy_rank: str  # the defeated enemy's rank letter


# input_builder callback: turns the typed arguments into the specialist's input text
def build_nested_input(options: dict) -> str:  # options: a plain dict at runtime
    params = options["params"]
    return f"Summon a shadow clone from {params['enemy_name']} (rank {params['enemy_rank']})."


structured_tool = shadow_clone_agent.as_tool(
    # tool_name: the name the model sees for this tool
    tool_name="summon_shadow_clone",
    # tool_description: what the tool does, so the model knows when to use it
    tool_description="Summon a shadow clone from a defeated enemy ninja.",
    # parameters: the typed shape of the tool's arguments
    parameters=ShadowCloneInput,
    # input_builder: builds the specialist's input text from those arguments
    input_builder=build_nested_input,
)
print("[structured schema]", structured_tool.params_json_schema)


# Demo: the typed arguments reach the specialist through input_builder
async def demo_structured_input() -> None:
    # Initialize the agent
    sage = Agent(name="Shadow Clone Sage", instructions="Use the tool.", tools=[structured_tool])

    # Script the model
    model = ScriptedModel(
        [
            [function_call("summon_shadow_clone", {"enemy_name": "Rogue Ninja", "enemy_rank": "B"}, call_id="c1")],
            [assistant_message("Summoning successful.")],
            [assistant_message("The Rogue Ninja shadow clone has joined your ranks.")],
        ]
    )

    # Run the Agent
    result = await Runner.run(sage, "go", run_config=RunConfig(model=model, workflow_name="e09-structured-demo"))
    # proves the typed arguments were accepted and the run finished normally
    print("[final_output]", result.final_output)


await demo_structured_input()
# Outside a notebook: asyncio.run(demo_structured_input())

## Change the result with `custom_output_extractor`

By default the tool's output is the specialist's last message, word for word. A
`custom_output_extractor` changes it before the calling agent sees it. Here it keeps
only the enemy's name.


In [ ]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunResult,  # everything a finished run gives back, like final_output and last_agent
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# custom_output_extractor callback: keeps only the enemy name from the specialist's reply
async def extract_enemy_name(run_result: RunResult) -> str:  # run_result: the specialist's finished run
    # "Summoning successful: Rogue Ninja (Jonin)." -> text after ":" and before "("
    return run_result.final_output.split(":")[-1].split("(")[0].strip()


extractor_tool = shadow_clone_agent.as_tool(
    # tool_name: the name the model sees for this tool
    tool_name="summon_shadow_clone",
    # tool_description: what the tool does, so the model knows when to use it
    tool_description="Summon a shadow clone from a defeated enemy ninja.",
    # custom_output_extractor: changes the specialist's output before the Sage sees it
    custom_output_extractor=extract_enemy_name,
)


# Demo: reads the tool output item to show the Sage got "Rogue Ninja", not the full sentence
async def demo_output_extractor() -> None:
    # Initialize the agent
    sage = Agent(name="Shadow Clone Sage", instructions="Use the tool.", tools=[extractor_tool])

    # Script the model
    model = ScriptedModel(
        [
            [function_call("summon_shadow_clone", {"input": "Rogue Ninja defeated"}, call_id="c1")],
            [assistant_message("Summoning successful: Rogue Ninja (Jonin).")],
            [assistant_message("Done.")],
        ]
    )

    # Run the Agent
    result = await Runner.run(sage, "go", run_config=RunConfig(model=model, workflow_name="e09-extractor-demo"))
    for item in result.new_items:
        if item.type == "tool_call_output_item":
            # proves the extractor changed the output to just the name
            print("[tool output after extraction]", item.output)


await demo_output_extractor()
# Outside a notebook: asyncio.run(demo_output_extractor())

## Hide a tool with `is_enabled` and watch the specialist with `on_stream`

- `is_enabled=False` hides the tool from the model completely, not just refuses calls. It is
  proven by capturing the tool list the model really sees.
- `on_stream=callback` runs for every streaming event of the specialist's own run, even when
  the outer run is not streamed. Like `input_builder`, it gets a _plain dict_ at runtime:
  `event["event"].type`, not `event.event.type`.


In [ ]:
from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from agents.testing import (
    ModelStep,  # one scripted step, made from your own function
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)

captured_tool_names: list[list[str]] = []  # the tool names each model call could see
stream_event_types: list[str] = []  # the event types the specialist's own run made


# ModelStep responder: saves exactly which tools the model call could see, then gives a fixed reply
async def capture_tools(call) -> ModelStep:  # call: the model request that is running now
    captured_tool_names.append([tool.name for tool in call.tools])
    return ModelStep(output=[assistant_message("ok")])


# on_stream callback: runs for every streaming event of the specialist's own inner run
def on_stream_callback(event: dict) -> None:  # event: a plain dict at runtime, so use dict keys only
    stream_event_types.append(event["event"].type)


disabled_tool = shadow_clone_agent.as_tool(
    # tool_name: the name the model sees for this tool
    tool_name="summon_shadow_clone",
    # tool_description: what the tool does, so the model knows when to use it
    tool_description="Summon a shadow clone from a defeated enemy ninja.",
    # is_enabled: False hides the tool from the model completely, instead of just saying no
    is_enabled=False,
)
streaming_tool = shadow_clone_agent.as_tool(
    # tool_name: the name the model sees for this tool
    tool_name="summon_shadow_clone",
    # tool_description: what the tool does, so the model knows when to use it
    tool_description="Summon a shadow clone from a defeated enemy ninja.",
    # on_stream: runs for every event that the specialist's own run makes
    on_stream=on_stream_callback,
)


# Demo: shows a disabled tool is missing from the model's tool list, and on_stream still sees inner events
async def demo_is_enabled_and_on_stream() -> None:
    # Initialize the agent
    disabled_sage = Agent(name="Shadow Clone Sage", instructions="Use the tool.", tools=[disabled_tool])
    # Script the model
    capture_model = ScriptedModel([ModelStep(responder=capture_tools)])
    # Run the Agent
    await Runner.run(disabled_sage, "go", run_config=RunConfig(model=capture_model, workflow_name="e09-is-enabled"))
    # proves is_enabled=False removes the tool from what the model sees, the list is empty
    print("[tools visible when is_enabled=False]", captured_tool_names)

    # Initialize the agent
    streaming_sage = Agent(name="Shadow Clone Sage", instructions="Use the tool.", tools=[streaming_tool])
    # Script the model
    stream_model = ScriptedModel(
        [
            [function_call("summon_shadow_clone", {"input": "Rogue Ninja"}, call_id="c1")],
            [assistant_message("Summoning successful.")],
            [assistant_message("Done.")],
        ]
    )
    # Run the Agent: a plain run, not a streamed one
    await Runner.run(streaming_sage, "go", run_config=RunConfig(model=stream_model, workflow_name="e09-on-stream"))
    # proves on_stream got the specialist's inner streaming events even though the outer run was not streamed
    print("[on_stream event types]", stream_event_types)


await demo_is_enabled_and_on_stream()
# Outside a notebook: asyncio.run(demo_is_enabled_and_on_stream())

## Run it live — optional, needs a real key

One real run with the same `as_tool()` + `custom_output_extractor` setup. Skipped
when there is no `OPENAI_API_KEY`.


In [ ]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # holds the instructions, tools, and model settings for one agent
    RunConfig,  # settings for one run, like model, workflow_name, and group_id
    RunResult,  # everything a finished run gives back, like final_output and last_agent
    Runner,  # runs an Agent: run, run_sync, or run_streamed
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# custom_output_extractor callback: keeps only the enemy name, or the raw text if the format differs
async def extract_name_live(run_result: RunResult) -> str:  # run_result: the specialist's finished run
    text = str(run_result.final_output)
    # a live model may not follow the format, so fall back instead of returning ""
    if ":" not in text or "(" not in text:
        return text
    return text.split(":")[-1].split("(")[0].strip()


# Demo: a real model calls the specialist as a tool and the Sage stays in control
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the specialist agent
    specialist = Agent(
        name="Shadow Clone Summoning Specialist",
        instructions="Reply with one line: 'Summoning successful: <enemy name> (<rank>).'",
    )

    # Initialize the agent
    sage = Agent(
        name="Shadow Clone Sage",
        instructions="Use summon_shadow_clone whenever an enemy ninja has just been defeated.",
        # tools: the tools this agent is allowed to call
        tools=[
            specialist.as_tool(
                # tool_name: the name the model sees for this tool
                tool_name="summon_shadow_clone",
                # tool_description: what the tool does, so the model knows when to use it
                tool_description="Summon a shadow clone from a defeated enemy ninja.",
                # custom_output_extractor: changes the helper agent's output before the main agent sees it
                custom_output_extractor=extract_name_live,
            )
        ],
    )

    # Run the Agent
    result = await Runner.run(
        sage,
        "Team 7 just defeated the Jonin-rank Rogue Ninja Zabuza at Mission-05.",
        # run_config: settings for just this run
        run_config=RunConfig(workflow_name="e09-live"),
    )

    # Inspect the result
    # proves as_tool() keeps control: last_agent is still the Sage, unlike a handoff
    print("[last_agent]", result.last_agent.name)
    for item in result.new_items:
        if item.type == "tool_call_output_item":
            # proves the extractor changed the specialist's real reply
            print("[tool output after extraction]", item.output)
    print("[final_output]", result.final_output)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

## Summary

- `as_tool()` is not a handoff: the specialist gets generated input and the original
  agent stays in control. The default schema is one `input: str`; `parameters=` and
  `input_builder=` give it a typed shape.
- `custom_output_extractor` changes the specialist's output before the caller sees it.
  `input_builder` and `on_stream` get plain dicts, and `is_enabled=False` hides the tool from the model.
